# Plantilla de experimento SLM en Colab

Duplica este notebook con un nombre descriptivo antes de editarlo. Define el objetivo, los datos, la hipotesis y las metricas del experimento aqui.

Ejecuta las celdas en orden con un runtime de Colab, tambien desde VS Code con kernel remoto. Publica antes los cambios de `src/slm` en GitHub: el runtime no ve los archivos de WSL. La arquitectura, el entrenamiento y la evaluacion reutilizables deben vivir en `src/slm`, no en este notebook.

Esta plantilla no ejecuta un entrenamiento por defecto. Completa las secciones de experimento antes de interpretar sus resultados.

## 1. Repositorio

Descargar o actualizar el codigo publicado. Se registra el commit completo para identificar la revision usada.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/miguel-conde/SLM.git"
REPO_DIR = Path("/content/SLM")

if REPO_DIR.exists():
    subprocess.run(
        ["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print(f"Repositorio: {REPO_DIR}\nCommit: {git_commit}")

## 2. Instalacion en el kernel de Colab

La instalacion normal evita el problema de imports observado con instalaciones editables en un kernel ya iniciado. `--no-deps` conserva el PyTorch del runtime con CUDA: **no instala ni valida las dependencias del proyecto**. El entorno real se muestra y registra abajo; puede diferir del declarado en `pyproject.toml`.

Si tu experimento necesita otras dependencias, declaralas en el proyecto y prepara un entorno compatible explicitamente. Tras actualizar el codigo, repite la instalacion; si ya has importado `slm`, reinicia el kernel y ejecuta de nuevo para evitar usar modulos antiguos en memoria.

In [ ]:
%pip install --no-deps .

## 3. Entorno y configuracion

Edita el nombre, la semilla y los parametros. Activa `REQUIRE_GPU` para detenerte si falta CUDA. Las semillas ayudan a repetir un experimento, pero no garantizan resultados identicos entre dispositivos o versiones.

In [ ]:
import json
import platform
import random
import tomllib
from datetime import datetime, timezone

import torch

import slm

EXPERIMENT_NAME = "my_experiment"
SEED = 42
REQUIRE_GPU = False
PARAMETERS = {}

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = (
    torch.cuda.get_device_name(0) if device.type == "cuda" else None
)
with Path("pyproject.toml").open("rb") as manifest_file:
    declared_dependencies = tomllib.load(manifest_file)["project"][
        "dependencies"
    ]

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}; CUDA de la rueda: {torch.version.cuda}")
print(f"Dispositivo: {device}; GPU: {gpu_name}")
print(f"Paquete slm: {slm.__file__}")
print(f"Dependencias declaradas (no validadas): {declared_dependencies}")
if REQUIRE_GPU and device.type != "cuda":
    raise RuntimeError("Este experimento requiere un runtime con GPU CUDA.")

## 4. Google Drive y rutas

Drive guarda datos y artefactos persistentes, no el repositorio. Usa `/content` como disco temporal para trabajo intensivo. Cada ejecucion crea su propio directorio de resultados y checkpoints.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/SLM")
if not (DRIVE_ROOT.parent).is_dir():
    raise RuntimeError("Google Drive no esta accesible en /content/drive.")

started_at = datetime.now(timezone.utc)
RUN_ID = started_at.strftime("%Y%m%d_%H%M%S_%f")
DATA_DIR = DRIVE_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = DRIVE_ROOT / "results" / EXPERIMENT_NAME / RUN_ID
CHECKPOINTS_DIR = DRIVE_ROOT / "checkpoints" / EXPERIMENT_NAME / RUN_ID
WORK_DIR = Path("/content/slm_work") / EXPERIMENT_NAME / RUN_ID

for directory in (
    RAW_DATA_DIR, PROCESSED_DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, WORK_DIR
):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Resultados: {RESULTS_DIR}")

## 5. Datos

Importa desde `slm` las funciones de carga y preparacion. Define la ruta de entrada en `RAW_DATA_DIR`; copia a `WORK_DIR` si interesa trabajar localmente. Registra la fuente/version de los datos en `PARAMETERS` y separa entrenamiento de validacion. No hay descarga ni datos implicitos en esta plantilla.

## 6. Modelo

Importa la arquitectura desde `slm`, instancia el modelo con `PARAMETERS` y muevelo a `device`. No copies aqui la definicion de clases.

## 7. Entrenamiento

Llama a la funcion reutilizable de entrenamiento de `slm`. Guarda checkpoints necesarios en `CHECKPOINTS_DIR` y evita escrituras frecuentes a Drive dentro del bucle de entrenamiento.

## 8. Evaluacion, graficos y generacion

Compara con el objetivo del experimento usando validacion separada. Completa `metrics` con valores Python serializables (por ejemplo, `loss.item()`), no tensores. Usa evaluacion/inferencia sin gradientes cuando proceda. Guarda figuras o muestras relevantes en `RESULTS_DIR`.

In [ ]:
metrics = {}

## 9. Persistencia del experimento

La celda exige metricas para no guardar una plantilla vacia como experimento completado. Guarda configuracion, commit y versiones reales; vuelve a leer el JSON para verificar su contenido. No garantiza reproducibilidad bit a bit ni certifica compatibilidad de dependencias.

In [ ]:
if not metrics:
    raise ValueError("Completa el experimento y sus metricas antes de guardar.")

report = {
    "experiment": EXPERIMENT_NAME,
    "run_id": RUN_ID,
    "started_at": started_at.isoformat(),
    "saved_at": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_commit,
    "python_version": platform.python_version(),
    "pytorch_version": str(torch.__version__),
    "cuda_version": torch.version.cuda,
    "device": str(device),
    "gpu": gpu_name,
    "declared_dependencies": declared_dependencies,
    "seed": SEED,
    "parameters": PARAMETERS,
    "metrics": metrics,
}
serialized_report = json.dumps(report, indent=2, allow_nan=False)
report_path = RESULTS_DIR / "experiment.json"
with report_path.open("x", encoding="utf-8") as report_file:
    report_file.write(serialized_report)
with report_path.open(encoding="utf-8") as report_file:
    saved_report = json.load(report_file)
if saved_report != json.loads(serialized_report):
    raise RuntimeError(f"El JSON leido no coincide: {report_path}")
print(f"Resultado guardado y verificado: {report_path}")

## 10. Conclusiones y siguientes pasos

- Resultado frente a la hipotesis:
- Limitaciones y errores observados:
- Siguiente experimento:

Antes de cerrar el runtime, comprueba que los artefactos necesarios estan en Drive. Implementa cambios de codigo en WSL, prueba y publica en GitHub; despues actualiza e instala de nuevo en Colab. No guardes tokens, datasets ni pesos en el notebook. Si VS Code avisa de que el archivo en disco es mas reciente, compara ambas versiones antes de sobrescribir.